In [4]:
import torch as T
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [14]:
class ReplyBuffer:
    def __init__(self,max_size, input_shape, n_action):

        self.max_size = max_size
        self.mem_ctr = 0
        self.state_memory = np.zero((max_size, *input_shape))
        self.next_state_memory = np.zero((max_size, *input_shape))
        self.reward_memory = np.zero((max_size, 1))
        self.action_memory = np.zero((max_size, n_action))
        self.terminate_memory = np.zero((max_size, 1))

    def store_transaction(self, state, action, reward, state_, done):

        index = self.mem_ctr % self.max_size

        self.state_memory[index] = state
        self.action_memory[index] = action
        self.reward_memory[index] = reward
        self.next_state_memory[index] = state_
        self.terminate_memory[index] = done

        self.mem_ctr +=1

    def sample_buffer(self, batch_size):
        max_mem = min(self.mem_cntr, self.mem_size)

        batch = np.random.choice(max_mem, batch_size)

        states = self.state_memory[batch]
        states_ = self.new_state_memory[batch]
        actions = self.action_memory[batch]
        rewards = self.reward_memory[batch]
        dones = self.terminal_memory[batch]

        return states, actions, rewards, states_, dones 
        

In [8]:
class CriticalNetwork(nn.Module):
    def __init__(self, beta, input_dim, fc1_dims, fc2_dims, n_actions):
        super(CriticalNetwork,self).__init__()
        # you know for Q(s,a) we need both s,a as input. its depend on structure creating so we add both inside input

        self.fc1 = nn.Linear(self.input_dim[0] + self.input_dim,fc1_dims)
        self.fc2 = nn.Linear(fc1_dims,fc2_dims)
        self.q1 = nn.Linear(fc2_dim,1)

        self.optim = optim.Adam(self.parameters(),lr = beta)

        self.device = ('cude:0' if T.cuda.is_available() else 'cpu')

        self.to(self.device)

    def forward(self, state, action):
        q1_action_value = self.fc1(T.cat([state, action], dim=1))
        q1_action_value = F.relu(q1_action_value)
        q1_action_value = self.fc2(q1_action_value)
        q1_action_value = F.relu(q1_action_value)

        q1 = self.q1(q1_action_value)
        return q1

    # you can add here
    #def save_to_checkpoint
    # load from check point

class ActorNetwork(nn.Module):
    def __init__(self, alpha, input_dims, fc1_dims, fc2_dims,
            n_actions):
        super(ActorNetwork, self).__init__()

        self.fc1 = nn.Linear(*self.input_dims, self.fc1_dims)
        self.fc2 = nn.Linear(self.fc1_dims, self.fc2_dims)
        self.mu = nn.Linear(self.fc2_dims, self.n_actions)

        self.optimizer = optim.Adam(self.parameters(), lr=alpha)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')

        self.to(self.device)

    def forward(self, state):
        prob = self.fc1(state)
        prob = F.relu(prob)
        prob = self.fc2(prob)
        prob = F.relu(prob)

        mu = T.tanh(self.mu(prob))

        return mu

***not sure but take a look on code later and see if we have two noise? one for generalization and for explore exploit***

***also see why we set done as zero in learn function??***

***its so strange to me that each one has its own loss, but optimize they are combined***

***we know that loss for actor is -Q -> but why it use critic1 for finding loss and why not2??***

In [ ]:
# so for agent we define d, that shows when we need to update actor network -> delay part -> 2 means every other learning step
# we also have warm up, let the agent has totally random action without noise 
# we have a noise in action, so we should have some max, min range to be sure its not out of range action.

# noise is here for explore exploit dillema



In [13]:
class Agent():
    def __init__(self, alpha, beta, input_dim, tau, env, gamma=0.99,
                update_actor_interval=2, warmup=1000,n_actions=2,max_size = 100000, 
                layer1_size=400, layer2_size=300, batch_size=100, noise=0.1):
        
        self.gamma = gamma
        self.tau = tau
        self.max_action = env.action_space.high
        self.min_action = env.action_space.low
        self.memory = ReplayBuffer(max_size, input_dims, n_actions)
        self.batch_size = batch_size
        self.learn_step_cntr = 0
        self.time_step = 0
        self.warmup = warmup
        self.n_actions = n_actions
        self.update_actor_iter = update_actor_interval


        self.actor = ActorNetwork(alpha, input_dims, layer1_size,
                        layer2_size, n_actions=n_actions)

        self.critic_1 = CriticNetwork(beta, input_dims, layer1_size,
                        layer2_size, n_actions=n_actions)
        self.critic_2 = CriticNetwork(beta, input_dims, layer1_size,
                        layer2_size, n_actions=n_actions)

        self.target_actor = ActorNetwork(alpha, input_dims, layer1_size,
                    layer2_size, n_actions=n_actions)
        self.target_critic_1 = CriticNetwork(beta, input_dims, layer1_size,
                layer2_size, n_actions=n_actions)
        self.target_critic_2 = CriticNetwork(beta, input_dims, layer1_size,
                layer2_size, n_actions=n_actions)

        self.noise = noise
        self.update_network_parameters(tau=1)

    def choose_action(self, observation):
        # check if still we are in warmup or not
        if self.time_step < self.warmup:
            # add some random action with scale
            mu = T.tensor(np.random.normal(scale=self.noise, size=(self.n_actions,)))
        else:
            state = T.tensor(observation, dtype=T.float32).to(self.actor.device)
            mu = self.actor(state).to(self.actor.device)

        mu_prime = mu + T.tensor(np.random.normal(scale=self.noise,size=(self.n_actions)))

        mu_prime = T.clamp(mu_prime, self.min_action[0], self.max_action[0])

        return mu_prime.cpu().detach().numpy()

    def remember(self, state, action, reward, new_state, done):
        self.memory.store_transition(state, action, reward, new_state, done)

    def learn(self):

        if self.memory.mem_ctr <= self.batch_size:
            return
            
        #states, actions, rewards, states_, dones 
        state, action, reward, state_, done = self.memory.sample_buffer(self.batch_size)
        
        state = T.tensor(state, dtype= T.float32).to(self.critic_1.device)
        action = T.tensor(action, dtype= T.float32).to(self.critic_1.device)
        reward = T.tensor(reward, dtype= T.float32).to(self.critic_1.device)
        state_ = T.tensor(state_, dtype= T.float32).to(self.critic_1.device)
        done = T.tensor(done, dtype= T.float32).to(self.critic_1.device)

        target_action = self.actor(state_)

        target_action = target_action + T.clamp(T.tensor(np.random.normal(scale=self.noise)),-0.5,0.5)
        target_action = T.clamp(target_action,self.min_action,self.max_action)

        q1_ = self.target_critic_1(state_,target_action)
        q2_ = self.target_critic_2(state_,target_action)

        q1 = self.target_critic_1(state,action)
        q2 = self.target_critic_2(state,action)

        q_value = T.min(q1_,q2_)


        # set q done
        q1_[done] = 0.0
        q2_[done] = 0.0

        target_q = reward + self.gamma * q_value

        self.critic_1.optimizer.zero_grad()
        self.critic_2.optimizer.zero_grad()

        critic_loss1 = F.mse_loss(target_q, q1)
        critic_loss2 = F.mse_loss(target_q, q2)

        total_loss = critic_loss1 + critic_loss2
        total_loss.backward()
    
        self.critic_1.optimizer.step()
        self.critic_2.optimizer.step()

        self.learn_step_cntr += 1


        # delay in update actor
        if self.learn_step_cntr % self.update_actor_iter != 0:
            return

        self.actor.optimizer.zero_grad()

        loss_q1_actor = self.critic_1(state, self.actor.forward(state))

        loss_actor = - T.mean(loss_q1_actor)

        actor_loss.backward()
        self.actor.optimizer.step()

        self.update_network_parameters()

    def update_network_parameters(self,tau = None):

        


        